# Final Paper Results

This notebook reads the filtered artifacts in `paper_results/`. It intentionally keeps only the selected tables and figures for the revision.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd()
if project_root.name == "comparisons":
    project_root = project_root.parent

paper_results = project_root / "paper_results"
tables_dir = paper_results / "tables"
figures_dir = paper_results / "figures"

assert tables_dir.exists(), f"Missing tables folder: {tables_dir}"
assert figures_dir.exists(), f"Missing figures folder: {figures_dir}"


## Tables Used in the Revision


In [ ]:
DATASET_ORDER = [
    "qsar fish toxicity",
    "concrete",
    "airfoil",
    "winered",
    "communities",
    "star",
    "abalone",
    "winewhite",
    "cycle",
    "electric",
    "meps19",
    "superconductivity",
    "homes",
    "protein",
    "WEC",
]

TABLE_FILES = {
    "SMIS": "result_qnn_smis.csv",
    "Interval length": "result_qnn_interval_length.csv",
    "Marginal coverage": "result_qnn_coverage.csv",
    "Outlier coverage (LOF)": "result_qnn_coverage_outliers.csv",
    "Outlier/inlier interval-length ratio (LOF)": "result_qnn_interval_ratio_outliers_inliers.csv",
    "Outlier coverage (Isolation Forest)": "result_qnn_coverage_outliers_isolation_forest.csv",
    "Outlier/inlier interval-length ratio (Isolation Forest)": "result_qnn_interval_ratio_outliers_inliers_isolation_forest.csv",
    "Worst scarcity coverage": "result_qnn_worst_scarcity_coverage.csv",
    "Fourth scarcity-score quartile coverage": "result_qnn_scarcity_q4_coverage.csv",
}

def dataset_key(label):
    return str(label).split(" (")[0]

def order_table(table):
    table = table.copy()
    order = {name: idx for idx, name in enumerate(DATASET_ORDER)}
    table["_order"] = table["Dataset"].map(lambda value: order.get(dataset_key(value), len(order)))
    return table.sort_values("_order").drop(columns="_order").reset_index(drop=True)

def read_table(filename):
    path = tables_dir / filename
    if not path.exists():
        raise FileNotFoundError(path)
    return order_table(pd.read_csv(path))

tables = {title: read_table(filename) for title, filename in TABLE_FILES.items()}

for title, table in tables.items():
    display(Markdown(f"### {title}"))
    display(table)


## LaTeX Sources


In [ ]:
TEX_FILES = {title: filename.replace(".csv", ".tex") for title, filename in TABLE_FILES.items()}

for title, filename in TEX_FILES.items():
    path = tables_dir / filename
    print(f"% {title}: {path.relative_to(project_root)}")
    print(path.read_text())
    print("\n")


## Figures Used in the Revision


In [ ]:
FIGURE_FILES = [
    "outlier_coverage_ratio_heatmap_lof.png",
    "outlier_coverage_ratio_selection_summary_lof.png",
    "outlier_coverage_ratio_heatmap_isolation_forest.png",
    "outlier_coverage_ratio_selection_summary_isolation_forest.png",
    "scarcity_q4_coverage_smis_heatmap.png",
    "scarcity_q4_coverage_smis_summary.png",
    "gamma_fixed_ablation_curves_main.png",
    "gamma_adaptive_ablation_curves_main.png",
]

for filename in FIGURE_FILES:
    path = figures_dir / filename
    assert path.exists(), f"Missing figure: {path}"
    print(path.relative_to(project_root))


## Quick Consistency Checks


In [ ]:
all_table_files = sorted(path.name for path in tables_dir.iterdir() if path.is_file())
all_figure_files = sorted(path.name for path in figures_dir.iterdir() if path.is_file())

forbidden_names = ["formatted", "drop" + "02", "qnn" + "_mc", "cat" + "boost", "w" + "sc"]
for forbidden in forbidden_names:
    matches = [name for name in all_table_files + all_figure_files if forbidden.lower() in name.lower()]
    assert not matches, f"Unexpected files containing {forbidden}: {matches}"

assert all(name.endswith((".csv", ".tex")) for name in all_table_files), all_table_files
assert all(name.endswith(".png") for name in all_figure_files), all_figure_files

print(f"Tables: {len(all_table_files)} files")
print(f"Figures: {len(all_figure_files)} files")
